# Empirical covariance spectra — temporary

Pass the `.pt` file produced by `compute_empirical_spectra.py`.
The file already determines the checkpoint, noise level, K, and τ.
The checkpoint and toy metadata referenced by the export must still be available.

`MANIFOLD_DIMS` maps every type in `manifold_dataset.py` to its intrinsic dimension
`R` and native extrinsic (embedding) dimension `M`, before embedding in the ambient
space. Average spectra are plotted for every type present in the export, pooling
repeated instances. Choose `MANIFOLD` for the individual-component plot.


In [17]:
from pathlib import Path
import json
import sys

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import torch

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))
from dalg.evaluation.toy_manifold_metrics import _associate_component_means
from dalg.data.manifold_dataset import MANIFOLD_NAMES, INTRINSIC_DIMS, EMBEDDING_DIMS

MANIFOLD_DIMS = {
    name: {"R": intrinsic_dim, "M": embedding_dim}
    for name, intrinsic_dim, embedding_dim in zip(
        MANIFOLD_NAMES, INTRINSIC_DIMS, EMBEDDING_DIMS, strict=True
    )
}

# SPECTRA_PATH = REPO / "dalg-cache/toy_manifolds_10types_1each_D128_30Keach_noise_sweep_seed0/noiseless/models/toy_noiseless_sweepk_em/hddc__toy_manifolds_10types_1each_d128_30keach_noiseless__l00__k500__q32__s42__37185291/empirical_spectra.pt"
# SPECTRA_PATH = REPO / "/u/dssc/zenocosini/decomposing-activations-local-geometry/dalg-cache/toy_manifolds_10types_1each_D128_30Keach_noise_sweep_seed0/noiseless/models/toy_noiseless_sweepk_em_bk/hddc__toy_manifolds_10types_1each_d128_30keach_noiseless__l00__k2000__q32__s42__3618d680/empirical_spectra.pt"
SPECTRA_PATH = REPO / "dalg-cache/toy_product_torus_6d_1each_D128_5M_noiseless_seed0/noiseless/models/toy_noiseless_sweepk_em/hddc__toy_product_torus_6d_1each_d128_5m_noiseless_seed0__l00__k2000__q32__s42__d68e8132/empirical_spectra.pt"
MANIFOLD = "sphere"
N_EIGENVALUES = 20
TORUS_EIGENVALUES = 30
SEPARATE_MANIFOLD = "product_torus_12d"
FIGURE_WIDTH = 1600

bundle = torch.load(SPECTRA_PATH, map_location="cpu", weights_only=True)
spectra = bundle["eigenvalues"].numpy()
source = bundle["source"]
config = json.loads(Path(source["config"]).read_text())
shard_dir = Path(source["shard_dir"])
shard_config = json.loads((shard_dir / "config.json").read_text())
metadata = torch.load(shard_dir / shard_config["manifold_metadata"], map_location="cpu", weights_only=True)
means = torch.load(source["checkpoint"], map_location="cpu", weights_only=True)["state_dict"]["mu"]

# Match the evaluator's nearest-manifold rule and saved distance cutoff.
run_spec_path = Path(source["config"]).parent / "run_spec.json"
cutoff = (json.loads(run_spec_path.read_text())["evaluation"].get("max_mean_to_manifold_distance")
          if run_spec_path.exists() else None)
association = _associate_component_means(means, metadata, max_mean_to_manifold_distance=cutoff)
component_ids_by_manifold = {}
for name in dict.fromkeys(m["type_name"] for m in metadata["manifolds"]):
    manifold_indices = [i for i, m in enumerate(metadata["manifolds"]) if m["type_name"] == name]
    ids = np.flatnonzero(
        bundle["valid"].numpy() & np.isin(association.manifold_indices.numpy(), manifold_indices)
    )
    if len(ids):
        component_ids_by_manifold[name] = ids
    else:
        print(f"Skipping {name}: no valid associated components.")
assert component_ids_by_manifold, "No valid components associated with any manifold."
print("Manifolds:", list(component_ids_by_manifold))

noise = shard_config["generator_config"]["noise_ratio"]
run_title = f"noise={noise} | K={bundle['K']} | τ={config.get('surgery_threshold', 'n/a')}"
x = np.arange(1, min(N_EIGENVALUES, spectra.shape[1]) + 1)

Manifolds: ['product_torus_6d']


## Average spectrum

One panel per manifold type, arranged three per row in a single figure, showing
the equal-weight mean of its associated
components' sorted spectra. Each component covariance still uses soft weights
over the full training split. This is not the spectrum of a pooled covariance.
Repeated instances of the same manifold type are pooled here.

`product_torus_12d` has a separate figure with its first 30 eigenvalues. The grid
shows the first 20 for the other types. All panels and the separate torus figure
use the same y-axis limits, computed across these displayed eigenvalues.
`FIGURE_WIDTH` controls the figure width in pixels; dots mark individual eigenvalues.
Set `LOG_SCALE = False` to show exact zeros; log plots omit them.


In [18]:
LOG_SCALE = True

mean_spectra = {
    name: spectra[ids].mean(axis=0)
    for name, ids in component_ids_by_manifold.items()
}

def plotted(values):
    return np.where(values > 0, values, np.nan) if LOG_SCALE else values


def shared_y_range(curves):
    values = np.concatenate(list(curves.values()))
    values = values[np.isfinite(values)]
    if LOG_SCALE:
        values = np.log10(values[values > 0])
    assert values.size, "No finite eigenvalues to plot on the selected scale."
    lower, upper = values.min(), values.max()
    padding = 0.05 * (upper - lower) if upper > lower else 0.05 * max(abs(upper), 1.0)
    return [lower - padding, upper + padding]


def spectrum_grid(curves, *, title, y_title, y_range, ncols=3):
    nrows = (len(curves) + ncols - 1) // ncols
    fig = make_subplots(
        rows=nrows, cols=ncols,
        specs=[
            [{} if row * ncols + col < len(curves) else None for col in range(ncols)]
            for row in range(nrows)
        ],
        subplot_titles=[
            f"{name} ({len(component_ids_by_manifold[name])} components)"
            for name in curves
        ],
        shared_xaxes="all", shared_yaxes="all",
        horizontal_spacing=0.07,
        vertical_spacing=0.20 / nrows,
        x_title="Eigenvalue index (largest first)",
        y_title=y_title,
    )
    for i, (name, values) in enumerate(curves.items()):
        plot_x = np.arange(1, len(values) + 1)
        row, col = i // ncols + 1, i % ncols + 1
        dims = MANIFOLD_DIMS[name]
        fig.add_trace(go.Scatter(
            x=plot_x, y=plotted(values), mode="lines+markers", name=name,
            marker=dict(size=6),
            line_color="royalblue",
        ), row=row, col=col)
        if dims["R"] <= plot_x[-1]:
            fig.add_vline(
                x=dims["R"], line_dash="dot", line_color="darkorange",
                annotation_text=f"R = {dims['R']}", annotation_position="top left",
                row=row, col=col,
            )
        if dims["M"] <= plot_x[-1]:
            fig.add_vline(
                x=dims["M"], line_dash="dot", line_color="seagreen",
                annotation_text=f"M = {dims['M']}", annotation_position="top right",
                row=row, col=col,
            )
    fig.update_xaxes(
        range=[0.5, max(len(values) for values in curves.values()) + 0.5],
        showticklabels=True, dtick=1 if ncols == 1 else 2,
    )
    fig.update_yaxes(
        type="log" if LOG_SCALE else "linear",
        range=y_range, showticklabels=True,
    )
    fig.update_layout(
        template="plotly_white", title=f"{title} | {run_title}",
        width=FIGURE_WIDTH, height=(450 if ncols == 1 else 320 * nrows) + 100,
        showlegend=False, hovermode="x unified",
        margin=dict(l=80, r=30, t=100, b=70),
    )
    return fig


def plot_average_spectra(curves, *, title, y_title):
    displayed = {
        name: values[:TORUS_EIGENVALUES if name == SEPARATE_MANIFOLD else N_EIGENVALUES]
        for name, values in curves.items()
    }
    y_range = shared_y_range(displayed)
    grid_curves = {name: values for name, values in displayed.items() if name != SEPARATE_MANIFOLD}
    if grid_curves:
        spectrum_grid(grid_curves, title=title, y_title=y_title, y_range=y_range).show()
    if SEPARATE_MANIFOLD in displayed:
        spectrum_grid(
            {SEPARATE_MANIFOLD: displayed[SEPARATE_MANIFOLD]},
            title=f"{title}: {SEPARATE_MANIFOLD}", y_title=y_title,
            y_range=y_range, ncols=1,
        ).show()


plot_average_spectra(mean_spectra, title="Average spectra", y_title="Eigenvalue")


## Normalized average spectrum

Divide each manifold's average spectrum by its largest eigenvalue:
$\bar{\lambda}_i / \max_j \bar{\lambda}_j$. Average across components first,
then normalize, so each curve starts at 1.

The grid has three panels per row and 20 eigenvalues per type. `product_torus_12d`
has a separate figure with 30 eigenvalues. Both use shared y-axis limits across
the normalized spectra, separate from the raw eigenvalue limits above.
The plots use the same `LOG_SCALE` setting; zero eigenvalues are omitted on a log scale.


In [19]:
normalized_mean_spectra = {}
for name, mean_spectrum in mean_spectra.items():
    lambda_max = mean_spectrum.max()
    assert lambda_max > 0, f"{name}: normalization requires a positive largest eigenvalue."
    normalized_mean_spectra[name] = mean_spectrum / lambda_max

plot_average_spectra(
    normalized_mean_spectra, title="Normalized average spectra", y_title="λᵢ / λₘₐₓ",
)


## Individual components

`MANIFOLD` selects the type for this plot; its `R` and `M` come from `MANIFOLD_DIMS`.
Set `COMPONENT_IDS = [3, 17]` using IDs printed by this cell, or leave it as `None`
to sample randomly from this manifold. Rerunning draws a new sample; set
`RANDOM_SEED` to an integer for repeatability.


In [20]:
assert MANIFOLD in component_ids_by_manifold, f"Choose a manifold from {list(component_ids_by_manifold)}"
component_ids = component_ids_by_manifold[MANIFOLD]
R, M = MANIFOLD_DIMS[MANIFOLD]["R"], MANIFOLD_DIMS[MANIFOLD]["M"]
title = f"{MANIFOLD} | {run_title}"
print("Selected component IDs:", component_ids.tolist())

COMPONENT_IDS = None
N_RANDOM = 5
RANDOM_SEED = None

if COMPONENT_IDS is None:
    selected = np.random.default_rng(RANDOM_SEED).choice(
        component_ids, size=min(N_RANDOM, len(component_ids)), replace=False)
else:
    selected = np.asarray(COMPONENT_IDS)
assert len(selected) and np.isin(selected, component_ids).all(), "Use IDs from the selected manifold."
print("Plotting components:", selected.tolist())

fig = go.Figure()
for k in selected:
    fig.add_trace(go.Scatter(
        x=x, y=plotted(spectra[k, :len(x)]), mode="lines+markers",
        marker=dict(size=6), name=f"Component {k}",
    ))
fig.update_layout(
    template="plotly_white",
    title=title,
    xaxis_title="Eigenvalue index (largest first)",
    xaxis_range=[0.5, len(x) + 0.5],
    yaxis_title="Eigenvalue",
    yaxis_type="log" if LOG_SCALE else "linear",
    width=FIGURE_WIDTH, height=450,
    hovermode="x unified",
)
if R <= x[-1]:
    fig.add_vline(
        x=R, line_dash="dot", line_color="darkorange",
        annotation_text=f"R = {R}", annotation_position="top left",
    )
if M <= x[-1]:
    fig.add_vline(
        x=M, line_dash="dot", line_color="seagreen",
        annotation_text=f"M = {M}", annotation_position="top right",
    )
fig.show()


AssertionError: Choose a manifold from ['product_torus_6d']